In [58]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%pip install pyspark

In [59]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create and config Spark
config = SparkConf().setAppName("bai05").setMaster("local[*]")
sparkContext = SparkContext.getOrCreate(conf=config)
sparkSession = SparkSession.builder.appName("bai05").getOrCreate()

In [60]:
data_path = "/content/drive/MyDrive/data/"

# Read file
movies_data = sparkContext.textFile(data_path + "movies.txt")

ratings1_data = sparkContext.textFile(data_path + "ratings_1.txt")
ratings2_data = sparkContext.textFile(data_path + "ratings_2.txt")

users_data = sparkContext.textFile(data_path + "users.txt")
occupation_data = sparkContext.textFile(data_path + "occupation.txt")

In [61]:
# Split occupation.txt
def splition_occupation(line):
    parts = line.split(',', 1)
    occupation_id = int(parts[0])
    occupation_name = parts[1] if len(parts) > 1 else f"Unknown"
    return (occupation_id, occupation_name)

occupations_map = occupation_data.map(splition_occupation)

occupations_dict = occupations_map.collectAsMap()


In [62]:
# Splition users.txt
def splition_user_occupation(line):
    parts = line.split(',')
    user_id = int(parts[0])
    gender = parts[1]
    age = int(parts[2])
    occupation_id = int(parts[3])
    zipcode = parts[4]
    return (user_id, occupation_id)

users_parsed = users_data.map(splition_user_occupation)

users_occupation_dict = users_parsed.collectAsMap()

In [63]:
def splition_rating_follow_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (user_id, rating)

ratings1_map = ratings1_data.map(splition_rating_follow_user)
ratings2_map = ratings2_data.map(splition_rating_follow_user)

# Combine 2 files
total_ratings = ratings1_map.union(ratings2_map)


In [64]:
def occupation_with_rating(row):
    user_id, rating = row
    occupation_id = users_occupation_dict.get(user_id, -100) # -100 when has no id occupation
    occupation_name = occupations_dict.get(occupation_id, "Unknown")
    return (occupation_name, rating)

ratings_and_occupation = total_ratings.map(occupation_with_rating)

# Filter unknown
filter_ratings = ratings_and_occupation.filter(lambda x: x[0] != "Unknown")

occupation_ratings_and_count = filter_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce key calculate total rating, number of rating for occupation
occupation_analysis = occupation_ratings_and_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))


In [ ]:
# calculate average rating for each occupation
def calculate_occupation_average(row):
    occupation, (sum_ratings, count) = row
    if sum_ratings is None or count is None or count == 0:
        avg = 0.0
        count = 0
    else:
        avg = sum_ratings / count
    return (occupation, (avg, count))

occupation_results = occupation_analysis.map(calculate_occupation_average)

result_occupations = occupation_results.collect()

for occupation, (avg, count) in result_occupations:
    print(f"{occupation} - AverageRating: {avg:.2f} (TotalRatings: {count})")

In [46]:

age_averages = {}

# Calculate average for each group
for group in groups_age:
    age_analysis = ratings_with_group_age[group].map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    age_averages[group] = age_analysis.map(lambda x: (x[0], x[1][0] / x[1][1]))  # (movie_id, average_rating)

# Calculate result for each film
movie_has_rating = filter_ratings.map(lambda x: x[0]).distinct()

list_movie_has_rating = movie_has_rating.collect()
list_results = []

for movie_id in list_movie_has_rating:
    movie_title = movies_dict.get(movie_id, f"No Name Movie")
    age_ratings = {}

    for group in groups_age:
        movie_ratings = age_averages[group].filter(lambda x: x[0] == movie_id).collect()
        if movie_ratings:
            age_ratings[group] = movie_ratings[0][1]  # average_point
        else:
            age_ratings[group] = None

    list_results.append((movie_id, movie_title, age_ratings))

# Print result
for movie_id, title, age_ratings in list_results:
    ratings_str = ", ".join([
        f"{group}: {age_ratings[group]:.2f}" if age_ratings[group] is not None else f"{group}: NA"
        for group in groups_age
    ])
    print(f"{title} - [{ratings_str}]")

E.T. the Extra-Terrestrial (1982) - [0-18: NA, 18-35: 3.56, 35-50: 3.83, 50+: 3.00]
Psycho (1960) - [0-18: NA, 18-35: 4.50, 35-50: 3.50, 50+: NA]
Gladiator (2000) - [0-18: NA, 18-35: 3.44, 35-50: 3.81, 50+: 3.50]
Fight Club (1999) - [0-18: NA, 18-35: 3.50, 35-50: 3.50, 50+: 3.50]
The Lord of the Rings: The Fellowship of the Ring (2001) - [0-18: NA, 18-35: 4.00, 35-50: 3.83, 50+: NA]
The Terminator (1984) - [0-18: NA, 18-35: 4.17, 35-50: 4.05, 50+: 3.75]
The Godfather: Part II (1974) - [0-18: NA, 18-35: 3.78, 35-50: 4.25, 50+: NA]
The Silence of the Lambs (1991) - [0-18: NA, 18-35: 3.00, 35-50: 3.25, 50+: NA]
Mad Max: Fury Road (2015) - [0-18: NA, 18-35: 3.36, 35-50: 3.64, 50+: NA]
Lawrence of Arabia (1962) - [0-18: NA, 18-35: 3.60, 35-50: 3.29, 50+: 4.50]
Sunset Boulevard (1950) - [0-18: NA, 18-35: 4.17, 35-50: 4.50, 50+: NA]
The Social Network (2010) - [0-18: NA, 18-35: 4.00, 35-50: 3.67, 50+: NA]
No Country for Old Men (2007) - [0-18: NA, 18-35: 3.81, 35-50: 3.94, 50+: 4.00]
The Lord